# Lab 8: Deploy Warehouse Assistant Agent to Amazon Bedrock AgentCore with SAP GenAI Hub

Welcome! In this tutorial, you'll learn how to deploy the intelligent warehouse operations agent from Lab 6 to Amazon Bedrock AgentCore Runtime while using SAP GenAI Hub for token consumption instead of direct Amazon Bedrock models.

## Business Scenario: Scalable Cloud Deployment

**Challenge:** After successfully building and testing our warehouse operations agent locally in Lab 6, we need to deploy it to a production-ready, scalable cloud environment where multiple users can access it simultaneously.

**Solution:** Deploy the agent to Amazon Bedrock AgentCore Runtime while maintaining SAP GenAI Hub integration for:
- Consistent token consumption through SAP's ecosystem
- Centralized governance and compliance
- Cost tracking through SAP's billing structure
- Enterprise-grade security and access controls

Note: If you have not set up your ai-core credentials yet, please follow notebook 00-load-sap-ai-core-credentials.

## Prerequisites

1. Completed Lab 6 - Warehouse operations agent
2. SAP AI Core credentials in your `~/.aicore/config.json` file
3. AWS credentials configured for your account
4. Amazon Bedrock AgentCore SDK installed
5. Strands Agents SDK installed
6. SAP GenAI Hub SDK installed

## Architecture Overview

In this lab, we'll deploy our multi-agent warehouse system to AgentCore while routing all LLM calls through SAP GenAI Hub:

```
┌─────────────────────────────────────────────────────────────┐
│                Amazon Bedrock AgentCore                      │
│  ┌────────────────────────────────────────────────────────┐ │
│  │              Docker Container                          │ │
│  │  ┌──────────────────────────────────────────────────┐ │ │
│  │  │  Warehouse Agent (from Lab 6)                    │ │ │
│  │  │  - Main orchestrator                             │ │ │
│  │  │  - Selector sub-agent                            │ │ │
│  │  │  - OData caller tool                             │ │ │
│  │  │  - ~/.aicore/config.json                         │ │ │
│  │  └──────────────────────────────────────────────────┘ │ │
│  └────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────┘
                            │
                            │ HTTPS (via SAP GenAI Hub SDK)
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                SAP AI Core / GenAI Hub                       │
│  ┌────────────────────────────────────────────────────────┐ │
│  │  Models Available:                                     │ │
│  │  - Amazon Nova (Pro, Lite, Micro)                     │ │
│  │  - Anthropic Claude (4.5 Sonnet, 4 Sonnet, 3 Haiku)  │ │
│  │  - Amazon Titan Models                                │ │
│  └────────────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────────────┘
```

## Step 1: Install Required Dependencies

Install all required packages from pyproject.toml.

In [1]:
# Install all dependencies from pyproject.toml
# %pip install .

## Step 2: Create the Agent Code File

Create a Python file that will run in the AgentCore container with the complete warehouse agent from Lab 6.

In [2]:
%%writefile warehouse_agent_agentcore.py
import os
import json
import logging
from strands import Agent, tool
from strands.multiagent.a2a.executor import StrandsA2AExecutor
from bedrock_agentcore.runtime import serve_a2a
from pathlib import Path
import yaml

from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from util.odata_tool import odata_caller

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def ensure_sap_api_key() -> str:
    """Ensure SAP API key is available in the environment."""
    cached = os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY")
    if cached:
        return cached
    else:
        raise ValueError("SAP_S4HANA_PUBLIC_CLOUD_KEY environment variable not set")

# Read SAP AI Core config
aicore_config_path = os.path.expanduser("~/.aicore/config.json")
if os.path.exists(aicore_config_path):
    with open(aicore_config_path) as f:
        config = json.loads(f.read())
    AI_API_URL = config["AICORE_BASE_URL"]
    AUTH_URL = config["AICORE_AUTH_URL"]
    CLIENT_ID = config["AICORE_CLIENT_ID"]
    CLIENT_SECRET = config["AICORE_CLIENT_SECRET"]
    RESOURCE_GROUP = config["AICORE_RESOURCE_GROUP"]
else:
    logger.info("No ~/.aicore/config.json found, reading from environment variables")
    AI_API_URL = os.environ.get("AICORE_BASE_URL", "")
    AUTH_URL = os.environ.get("AICORE_AUTH_URL", "")
    CLIENT_ID = os.environ.get("AICORE_CLIENT_ID", "")
    CLIENT_SECRET = os.environ.get("AICORE_CLIENT_SECRET", "")
    RESOURCE_GROUP = os.environ.get("AICORE_RESOURCE_GROUP", "")

# Initialize model
model = SAPGenAIHubModel(model_id="anthropic--claude-4.5-sonnet")

def load_openapi_specs(path="./assets/knowledgebase"):
    specs = []
    for file in Path(path).glob("*.y*ml"):
        with open(file, "r") as f:
            try:
                data = yaml.safe_load(f)
                servers = data.get("servers", [])
                base_urls = []
                for s in servers:
                    url = s.get("url")
                    desc = s.get("description", "")
                    if url:
                        base_urls.append({"url": url, "description": desc})
                summary = {
                    "file": file.name,
                    "title": data.get("info", {}).get("title"),
                    "description": data.get("info", {}).get("description"),
                    "paths": list(data.get("paths", {}).keys()),
                    "base_urls": base_urls or [{"url": "/", "description": "Default"}],
                }
                specs.append(summary)
            except Exception as e:
                logger.warning(f"Error parsing {file}: {e}")
    return specs


def specs_to_prompt_string(specs):
    prompt_parts = []
    for spec in specs:
        base_url_str = ", ".join(
            f"{url_info['url']} ({url_info['description']})"
            for url_info in spec["base_urls"]
        )
        paths_str = ", ".join(spec["paths"])
        part = (
            f"API Spec: {spec['file']}\n"
            f"Title: {spec['title']}\n"
            f"Description: {spec['description']}\n"
            f"Base URLs: {base_url_str}\n"
            f"Endpoints: {paths_str}\n"
        )
        prompt_parts.append(part)
    return "\n---\n".join(prompt_parts)


specs = load_openapi_specs()
if not specs:
    logger.warning("No OpenAPI specs found in ./assets/knowledgebase")
specs_prompt_string = specs_to_prompt_string(specs)

SELECTOR_SYSTEM_PROMPT = f"""
You are an API selection subagent.
Given a user query and the following list of OpenAPI specs, each with a title, description, base URLs, and available endpoints:

{specs_prompt_string}

Your task is to identify which API and specific endpoint is most appropriate to fulfill the user query.
Provide clear reasoning for your choice, referencing the API spec details provided.
If no suitable API is found, explain why.

Make sure to reply with the sandbox BASE URL.
"""

# Create selector agent
selector_agent = Agent(system_prompt=SELECTOR_SYSTEM_PROMPT, model=model)


@tool
def SelectorAPIAgentAsATool(query: str) -> str:
    """Analyzes available OpenAPI specs and selects the most appropriate API and endpoint.

    Args:
        query: User query describing what they want to accomplish

    Returns:
        Agent response with API selection recommendation
    """
    return selector_agent(query).message


@tool
def fetch_sap_api_key() -> str:
    """Fetches and caches the SAP API key for use with OData calls.

    Returns:
        Confirmation that the API key is available.
    """
    try:
        ensure_sap_api_key()
        return "SAP API key is ready. Use auth_env_var='SAP_S4HANA_PUBLIC_CLOUD_KEY' with odata_caller."
    except Exception as e:
        return f"Failed to fetch SAP API key: {e}"


warehouse_agent_prompt = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750). 
You have access to real-time SAP warehouse data through dynamic OData API exploration capabilities.

IMPORTANT - MANDATORY WORKFLOW:
You MUST follow these steps in order for EVERY user query:

STEP 0 (REQUIRED): Call fetch_sap_api_key to ensure the API key is available.
STEP 1 (REQUIRED): Call SelectorAPIAgentAsATool with the user's query to determine which API and endpoint to use.
STEP 2: Use the base URL and endpoint returned by the selector to call odata_caller with the $metadata endpoint to understand the data model.
STEP 3: Construct the appropriate OData query using odata_caller based on what you learned.

NEVER skip Steps 0 and 1.

AUTHENTICATION:
When calling odata_caller, ALWAYS use these auth parameters:
- auth_type: "api_key"  
- auth_env_var: "SAP_S4HANA_PUBLIC_CLOUD_KEY"

EXAMPLE odata_caller usage:
```
odata_caller(
    base_url="https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata/sap/",
    endpoint="API_WAREHOUSE_STOCK_SRV/WarehouseStockProducts",
    operation="get",
    odata_params={"$filter": "Product eq 'WM-AN02'", "$top": "10"},
    auth_type="api_key",
    auth_env_var="SAP_S4HANA_PUBLIC_CLOUD_KEY"
)
```

PRODUCT KNOWLEDGE (can be expanded through discovery):
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Explain your discovery process when exploring new data
- Provide specific, actionable insights with quantitative data
- Always show the URL you constructed so the user is informed
- Do not use emojis
"""

# Create the warehouse agent
warehouse_agent = Agent(
    name="Warehouse Ops Agent",
    description="An expert Warehouse Operations Manager that queries SAP OData APIs for real-time warehouse stock, product, and supply chain data.",
    model=model,
    tools=[fetch_sap_api_key, SelectorAPIAgentAsATool, odata_caller],
    system_prompt=warehouse_agent_prompt,
)

if __name__ == "__main__":
    serve_a2a(StrandsA2AExecutor(warehouse_agent))

Overwriting warehouse_agent_agentcore.py


## Step 3: Create Requirements File

Create the requirements file with AgentCore and SAP GenAI Hub dependencies.

In [ ]:
%%writefile requirements.txt
# AgentCore requirements
aioboto3
boto3
duckduckgo-search>=8.0.4
sap-ai-sdk-gen[all]>=6.10.0
ipykernel>=6.29.5
langchain>=0.3.25
langgraph>=0.3.25
strands-agents[a2a]>=1.5.0
strands-agents-builder>=0.1.8
strands-agents-tools[a2a_client]==0.2.0
uvicorn>=0.34.3    
pydantic
bedrock-agentcore[a2a]>=1.10.0
bedrock-agentcore-starter-toolkit

Overwriting requirements.txt


## Step 4: Create SAP AI Core Configuration File

This is the critical step for enabling SAP GenAI Hub token consumption in AgentCore.

In [4]:
import json
import os

# Load the existing config from ~/.aicore/config.json
config_path = os.path.expanduser('~/.aicore/config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Write the config file that will be copied into the Docker container
    with open('config.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    print("SAP AI Core configuration file created: config.json")
    print("This file will be copied into the AgentCore container for GenAI Hub access.")
else:
    print("Error: ~/.aicore/config.json not found!")
    print("Please ensure you have your SAP AI Core credentials configured.")

SAP AI Core configuration file created: config.json
This file will be copied into the AgentCore container for GenAI Hub access.


## Step 5: Create a Cognito User Pool 

Create a Cognito user pool for Agentcore Runtime authentication

In [9]:
USER_POOL_ID=!aws cognito-idp create-user-pool \
  --pool-name "warehouse-ops-a2a" \
  --policies "PasswordPolicy={MinimumLength=8,RequireUppercase=true,RequireLowercase=true,RequireNumbers=true,RequireSymbols=false}" \
  --auto-verified-attributes email \
  --username-attributes email \
  --query 'UserPool.Id' \
  --output text

%env USER_POOL_ID={USER_POOL_ID[0]}


env: USER_POOL_ID=us-west-2_MCrww684F


## Step 6: Create a Cognito App Client

Create a Cognito app client for Joule Authentication

In [27]:
import os
import subprocess

# Create the Cognito user pool client and capture CLIENT_ID
result = subprocess.run(
    [
        "aws", "cognito-idp", "create-user-pool-client",
        "--user-pool-id", os.getenv("USER_POOL_ID"),
        "--client-name", "joule-client",
        "--explicit-auth-flows", 
        "ALLOW_USER_PASSWORD_AUTH", 
        "ALLOW_REFRESH_TOKEN_AUTH", 
        "ALLOW_USER_SRP_AUTH", 
        "ALLOW_CUSTOM_AUTH",
        "--generate-secret",
        "--query", "UserPoolClient.ClientId",
        "--output", "text"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    client_id = result.stdout.strip()
    os.environ["CLIENT_ID"] = client_id
    print(f"✓ CLIENT_ID created: {client_id}")
else:
    print(f"✗ Error creating client: {result.stderr}")

# Extract region from USER_POOL_ID
user_pool_id = os.getenv("USER_POOL_ID")
region = user_pool_id.split('_')[0]

# Set environment variables
os.environ["REGION"] = region
os.environ["DISCOVERY_URL"] = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

# Verify all environment variables are set
print(f"\n✓ Environment variables set:")
print(f"  CLIENT_ID: {os.getenv('CLIENT_ID')}")
print(f"  USER_POOL_ID: {os.getenv('USER_POOL_ID')}")
print(f"  REGION: {os.getenv('REGION')}")
print(f"  DISCOVERY_URL: {os.getenv('DISCOVERY_URL')}")

✓ CLIENT_ID created: 4ssveq0p8pu7nrj7smqgmugqnp

✓ Environment variables set:
  CLIENT_ID: 4ssveq0p8pu7nrj7smqgmugqnp
  USER_POOL_ID: us-west-2_MCrww684F
  REGION: us-west-2
  DISCOVERY_URL: https://cognito-idp.us-west-2.amazonaws.com/us-west-2_MCrww684F/.well-known/openid-configuration


## Step 6b: Create Resource Server and Enable OAuth on App Client

Create the Cognito resource server that defines the `warehouse-ops-a2a/invoke` scope, then enable `client_credentials` OAuth flow on the app client.

In [ ]:
import subprocess
import json
import os

user_pool_id = os.getenv("USER_POOL_ID")
client_id = os.getenv("CLIENT_ID")
region = os.getenv("REGION")

# Step 1: Create resource server
rs_result = subprocess.run(
    [
        "aws", "cognito-idp", "create-resource-server",
        "--user-pool-id", user_pool_id,
        "--identifier", "warehouse-ops-a2a",
        "--name", "Warehouse Ops Agent",
        "--scopes", "ScopeName=invoke,ScopeDescription=Invoke warehouse ops agent",
        "--region", region
    ],
    capture_output=True, text=True
)

if rs_result.returncode == 0:
    print("✓ Resource server created: warehouse-ops-a2a")
    print(json.dumps(json.loads(rs_result.stdout), indent=2))
else:
    # Already exists is acceptable — scope is still usable
    if "already exists" in rs_result.stderr:
        print("✓ Resource server already exists — skipping creation")
    else:
        print(f"✗ Error creating resource server: {rs_result.stderr}")

# Step 2: Enable client_credentials OAuth flow and attach scope
oauth_result = subprocess.run(
    [
        "aws", "cognito-idp", "update-user-pool-client",
        "--user-pool-id", user_pool_id,
        "--client-id", client_id,
        "--region", region,
        "--allowed-o-auth-flows", "client_credentials",
        "--allowed-o-auth-scopes", "warehouse-ops-a2a/invoke",
        "--allowed-o-auth-flows-user-pool-client"
    ],
    capture_output=True, text=True
)

if oauth_result.returncode == 0:
    print("\n✓ OAuth client_credentials flow enabled with scope: warehouse-ops-a2a/invoke")
else:
    print(f"✗ Error updating client: {oauth_result.stderr}")

# Step 3: Create Cognito hosted UI domain (required for /oauth2/token endpoint)
domain_name = f"warehouse-ops-{user_pool_id.split('_')[1].lower()}"
domain_result = subprocess.run(
    [
        "aws", "cognito-idp", "create-user-pool-domain",
        "--user-pool-id", user_pool_id,
        "--domain", domain_name,
        "--region", region
    ],
    capture_output=True, text=True
)

if domain_result.returncode == 0:
    print(f"\n✓ Cognito domain created: {domain_name}")
else:
    if "already exists" in domain_result.stderr or "Domain already" in domain_result.stderr:
        print(f"\n✓ Cognito domain already exists: {domain_name}")
    else:
        print(f"✗ Error creating domain: {domain_result.stderr}")

os.environ["COGNITO_DOMAIN"] = f"https://{domain_name}.auth.{region}.amazoncognito.com"
os.environ["OAUTH_SCOPE"] = "warehouse-ops-a2a/invoke"

print(f"\n✓ Token endpoint: {os.getenv('COGNITO_DOMAIN')}/oauth2/token")
print(f"✓ Scope: {os.getenv('OAUTH_SCOPE')}")

## Step 6c: Generate Bearer Token

Fetch the app client secret and exchange it for a bearer token using the Cognito `client_credentials` OAuth flow.

In [ ]:
import subprocess
import json
import os
import time
import requests

user_pool_id = os.getenv("USER_POOL_ID")
client_id = os.getenv("CLIENT_ID")
region = os.getenv("REGION")
cognito_domain = os.getenv("COGNITO_DOMAIN")
oauth_scope = os.getenv("OAUTH_SCOPE")

# Step 1: Retrieve the client secret
secret_result = subprocess.run(
    [
        "aws", "cognito-idp", "describe-user-pool-client",
        "--user-pool-id", user_pool_id,
        "--client-id", client_id,
        "--region", region,
        "--query", "UserPoolClient.ClientSecret",
        "--output", "text"
    ],
    capture_output=True, text=True
)

if secret_result.returncode != 0:
    raise RuntimeError(f"Failed to retrieve client secret: {secret_result.stderr}")

client_secret = secret_result.stdout.strip()
print(f"✓ Client secret retrieved: {client_secret[:6]}***")

# Step 2: Exchange client credentials for a bearer token
# Cognito domain DNS can take 1-2 minutes to propagate after creation — retry with backoff
token_url = f"{cognito_domain}/oauth2/token"
max_attempts = 12
wait_seconds = 15
token_data = None

for attempt in range(1, max_attempts + 1):
    try:
        print(f"  Attempt {attempt}/{max_attempts}: POST {token_url}")
        token_response = requests.post(
            token_url,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={
                "grant_type": "client_credentials",
                "scope": oauth_scope
            },
            auth=(client_id, client_secret),
            timeout=10
        )
        if token_response.status_code == 200:
            token_data = token_response.json()
            break
        else:
            print(f"  HTTP {token_response.status_code}: {token_response.text}")
    except (requests.ConnectionError, requests.Timeout) as e:
        print(f"  DNS not ready yet: {e.__class__.__name__} — waiting {wait_seconds}s...")

    if attempt < max_attempts:
        time.sleep(wait_seconds)

if token_data is None:
    raise RuntimeError(
        f"Failed to obtain token after {max_attempts} attempts. "
        "Cognito domain DNS may still be propagating — wait a minute and re-run this cell."
    )

bearer_token = token_data["access_token"]
os.environ["BEARER_TOKEN"] = bearer_token

print(f"\n✓ Token type   : {token_data['token_type']}")
print(f"✓ Expires in   : {token_data['expires_in']} seconds")
print(f"✓ Bearer token : {bearer_token[:40]}...")

## Step 7: Configure AgentCore Runtime Deployment

Configure the AgentCore Runtime deployment.

In [ ]:
import os
from pathlib import Path
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

# Clear stale cached agent state so configure() creates a new agent
# instead of trying to update a deleted one
stale_config = Path.home() / ".bedrock_agentcore.yaml"
if stale_config.exists():
    stale_config.unlink()
    print("✓ Cleared stale agent config: ~/.bedrock_agentcore.yaml")
else:
    print("✓ No stale agent config found")

# Initialize fresh
boto_session = Session()
region = boto_session.region_name

# Get environment variables
client_id = os.getenv("CLIENT_ID")
user_pool_id = os.getenv("USER_POOL_ID")
discovery_url = os.getenv("DISCOVERY_URL")

print(f"\nConfiguring agent deployment in region: {region}")

agentcore_runtime = Runtime()
agent_name = "warehouse_ops_agent_sap_genai_hub"

response = agentcore_runtime.configure(
    entrypoint="warehouse_agent_agentcore.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    protocol="A2A",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
            # Note: allowedAudience is intentionally omitted —
            # Cognito client_credentials tokens have no 'aud' claim
        }
    },
    non_interactive=True
)

print("✅ Agent configured successfully")
print(f"  Protocol : A2A")
print(f"  Region   : {region}")
print(f"  Client ID: {client_id}")
print(response)

## Step 8: Modify the Generated Dockerfile

**Critical Step**: Modify the generated Dockerfile to include the SAP GenAI Hub configuration and ensure all dependencies are available in the container.

This step implements:
1. SAP GenAI Hub configuration using `AICORE_HOME` environment variable (per SAP documentation)
2. Essential dependencies (util/ and assets/ directories)

**Updated**: Uses proper SAP GenAI Hub SDK configuration pattern with `AICORE_HOME`.

In [45]:
# Read the generated Dockerfile
with open('Dockerfile', 'r') as f:
    dockerfile_content = f.read()

# Find the insertion point (before CMD instruction)
lines = dockerfile_content.split('\n')
cmd_index = -1
for i, line in enumerate(lines):
    if line.strip().startswith('CMD'):
        cmd_index = i
        break

if cmd_index > -1:
    # Insert all required configuration before CMD
    config_lines = [
        "",
        "# Copy util directory (required for SAP GenA Hub model and OData tool)",
        "COPY util/ ./util/",
        "",
        "# Copy assets directory (required for OpenAPI knowledgebase)",
        "COPY assets/ ./assets/",
        "",
        "# Copy the config.json from your local machine",
        "COPY config.json /app/.aicore/config.json",
        "",
        "# Set AICORE_HOME environment variable for SAP GenAI Hub SDK",
        "ENV AICORE_HOME=/app/.aicore",
        ""
    ]
    
    # Insert the new lines before CMD
    modified_lines = lines[:cmd_index] + config_lines + lines[cmd_index:]
    modified_content = '\n'.join(modified_lines)
    
    # Write the modified Dockerfile
    with open('Dockerfile', 'w') as f:
        f.write(modified_content)
    
    print(" Dockerfile successfully modified with all required dependencies!")
    print("\n Added the following for AgentCore containerization:")
    print("  COPY util/ ./util/                    # SAP GenAI Hub model & OData tool")
    print("  COPY assets/ ./assets/                # OpenAPI knowledgebase")
    print("  RUN mkdir -p /app/.aicore             # GenAI Hub credentials directory")
    print("  COPY config.json /app/.aicore/config.json  # GenAI Hub credentials")
    print("  ENV AICORE_HOME=/app/.aicore          # SAP GenAI Hub SDK directory")
    print("\n All dependencies now available in AgentCore container!")
else:
    print(" Could not find CMD instruction in Dockerfile")

 Dockerfile successfully modified with all required dependencies!

 Added the following for AgentCore containerization:
  COPY util/ ./util/                    # SAP GenAI Hub model & OData tool
  COPY assets/ ./assets/                # OpenAPI knowledgebase
  RUN mkdir -p /app/.aicore             # GenAI Hub credentials directory
  COPY config.json /app/.aicore/config.json  # GenAI Hub credentials
  ENV AICORE_HOME=/app/.aicore          # SAP GenAI Hub SDK directory

 All dependencies now available in AgentCore container!


## Step 9: Launch Agent to AgentCore Runtime

Deploy the agent to Amazon Bedrock AgentCore Runtime.

In [47]:
# Launch the agent with SAP API key environment variable
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get the SAP S/4HANA Public Cloud API key from environment
sap_api_key = os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY")
if not sap_api_key:
    import getpass
    sap_api_key = getpass.getpass("Please enter your SAP_S4HANA_PUBLIC_CLOUD_KEY: ")


launch_result = agentcore_runtime.launch(
    env_vars={
        "SAP_S4HANA_PUBLIC_CLOUD_KEY": sap_api_key
    }
)

print("✅ Deployment completed!")
print(f"Agent ID: {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)


   • Build ARM64 containers in the cloud with CodeBuild


   • No local Docker required


💡 Available deployment modes:


   • runtime.launch()                           → CodeBuild (current)


...

   /aws/bedrock-agentcore/runtimes/warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu-DEFAULT --log-stream-name-prefix "2026/06/05/\[runtime-logs]"


   /aws/bedrock-agentcore/runtimes/warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu-DEFAULT --log-stream-names "otel-rt-logs"


💡 Tail logs with: aws logs tail /aws/bedrock-agentcore/runtimes/warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu-DEFAULT --log-stream-name-prefix "2026/06/05/\[runtime-logs]" --follow


💡 Or view recent logs: aws logs tail /aws/bedrock-agentcore/runtimes/warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu-DEFAULT --log-stream-name-prefix "2026/06/05/\[runtime-logs]" --since 1h


✅ Deployment completed!
Agent ID: warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu
Agent ARN: arn:aws:bedrock-agentcore:us-west-2:543030844330:runtime/warehouse_ops_agent_sap_genai_hub-cr9Rzg7dAu


## Step 10: Check AgentCore Runtime Status

Monitor the deployment status until it's ready.

In [48]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Initial status: {status}")

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)

print(f"\nFinal status: {status}")
status

Retrieved Bedrock AgentCore status for: warehouse_ops_agent_sap_genai_hub


Initial status: READY

Final status: READY


'READY'

## Step 11: Invoke AgentCore A2A Agent via JSON-RPC 2.0

Call the deployed A2A agent directly over HTTP using the bearer token from Step 6c and a standard JSON-RPC 2.0 `message/send` payload. This is the same protocol an external client (e.g. Joule) would use.

In [ ]:
import boto3
import json
import os
import uuid
from urllib.parse import quote
import requests

agent_arn    = launch_result.agent_arn
bearer_token = os.getenv("BEARER_TOKEN")
region       = os.getenv("REGION")

# Build the correct invoke URL using the confirmed pattern from boto3 inspection:
# https://bedrock-agentcore.<region>.amazonaws.com/runtimes/<url-encoded-arn>/invocations?qualifier=DEFAULT
encoded_arn  = quote(agent_arn, safe="")
invoke_url   = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
print(f"Invoke URL: {invoke_url}")

# ── JSON-RPC 2.0 payload ──────────────────────────────────────────────────────
def make_jsonrpc_payload(text: str) -> dict:
    return {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
                "messageId": str(uuid.uuid4())
            }
        }
    }

# ── Invoke ────────────────────────────────────────────────────────────────────
queries = [
    "What stock do we have for WM-AN02?",
    "Can we fulfill an order for 50 units of WM-AN01 sensors?",
]

for query in queries:
    payload = make_jsonrpc_payload(query)
    print(f"\n{'─'*60}")
    print(f"Query  : {query}")

    response = requests.post(
        invoke_url,
        headers={
            "Authorization": f"Bearer {bearer_token}",
            "Content-Type": "application/json"
        },
        json=payload,
        timeout=60
    )

    print(f"HTTP   : {response.status_code}")
    if response.status_code != 200:
        print(f"Error  : {response.text}")
        continue

    try:
        rpc_response = response.json()
        parts = rpc_response.get("result", {}).get("parts", [])
        answer = " ".join(p.get("text", "") for p in parts if p.get("kind") == "text")
        print(f"Answer : {answer if answer else json.dumps(rpc_response, indent=2)}")
    except Exception as e:
        print(f"Raw    : {response.text}")
        print(f"Error  : {e}")

## Step 12: Let's gather all the details for the next module 

Let's gather all the required details for the next module Joule Integration

In [ ]:
import os
import subprocess
from urllib.parse import quote

# ── Collect all Joule integration variables ───────────────────────────────────
region       = os.getenv("REGION")
client_id    = os.getenv("CLIENT_ID")
cognito_domain = os.getenv("COGNITO_DOMAIN")
oauth_scope  = os.getenv("OAUTH_SCOPE")
agent_arn    = launch_result.agent_arn

# 1. Invoke URL
encoded_arn = quote(agent_arn, safe="")
invoke_url  = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

# 2. Client ID (already in env)

# 3. Client Secret — fetch live from Cognito
user_pool_id = os.getenv("USER_POOL_ID")
secret_result = subprocess.run(
    [
        "aws", "cognito-idp", "describe-user-pool-client",
        "--user-pool-id", user_pool_id,
        "--client-id", client_id,
        "--region", region,
        "--query", "UserPoolClient.ClientSecret",
        "--output", "text"
    ],
    capture_output=True, text=True
)
client_secret = secret_result.stdout.strip() if secret_result.returncode == 0 else "ERROR: could not retrieve"

# 4. Scope
scope = oauth_scope

# 5. Token Service URL
token_service_url = f"{cognito_domain}/oauth2/token"

# ── Print summary ─────────────────────────────────────────────────────────────
print("=" * 65)
print("  JOULE INTEGRATION VARIABLES")
print("=" * 65)
print(f"  1. Invoke URL        : {invoke_url}")
print(f"  2. Client ID         : {client_id}")
print(f"  3. Client Secret     : {client_secret}")
print(f"  4. Scope             : {scope}")
print(f"  5. Token Service URL : {token_service_url}")
print("=" * 65)

## Step 13: Cleanup (Optional)

Clean up the AgentCore Runtime when no longer needed.

In [ ]:
# ── Cleanup: uncomment and run to delete all resources created in this notebook ──
import boto3
import os

# region       = os.getenv("REGION")
# user_pool_id = os.getenv("USER_POOL_ID")
# client_id    = os.getenv("CLIENT_ID")
# agent_id     = launch_result.agent_id
# ecr_repo     = f"bedrock-agentcore-{launch_result.agent_id.split('-')[0]}"

# agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
# cognito_client           = boto3.client("cognito-idp",               region_name=region)
# ecr_client               = boto3.client("ecr",                       region_name=region)
# iam_client               = boto3.client("iam")

# ── 1. Delete AgentCore runtime ───────────────────────────────────────────────
# agentcore_control_client.delete_agent_runtime(agentRuntimeId=agent_id)
# print(f"✓ Deleted AgentCore runtime: {agent_id}")

# ── 2. Delete ECR repository ──────────────────────────────────────────────────
# ecr_repo_name = f"bedrock-agentcore-warehouse_ops_agent_sap_genai_hub"
# ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
# print(f"✓ Deleted ECR repository: {ecr_repo_name}")

# ── 3. Delete Cognito domain ──────────────────────────────────────────────────
# domain_name = f"warehouse-ops-{user_pool_id.split('_')[1].lower()}"
# cognito_client.delete_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
# print(f"✓ Deleted Cognito domain: {domain_name}")

# ── 4. Delete Cognito resource server ────────────────────────────────────────
# cognito_client.delete_resource_server(UserPoolId=user_pool_id, Identifier="warehouse-ops-a2a")
# print("✓ Deleted Cognito resource server: warehouse-ops-a2a")

# ── 5. Delete Cognito app client ──────────────────────────────────────────────
# cognito_client.delete_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)
# print(f"✓ Deleted Cognito app client: {client_id}")

# ── 6. Delete Cognito user pool ───────────────────────────────────────────────
# cognito_client.delete_user_pool(UserPoolId=user_pool_id)
# print(f"✓ Deleted Cognito user pool: {user_pool_id}")

# ── 7. Delete AgentCore execution IAM role ────────────────────────────────────
# runtime_role = f"AmazonBedrockAgentCoreSDKRuntime-{region}-{agent_id.split('-')[-1]}"
# for policy in iam_client.list_attached_role_policies(RoleName=runtime_role)["AttachedPolicies"]:
#     iam_client.detach_role_policy(RoleName=runtime_role, PolicyArn=policy["PolicyArn"])
# iam_client.delete_role(RoleName=runtime_role)
# print(f"✓ Deleted IAM role: {runtime_role}")

# ── 8. Delete CodeBuild execution IAM role ────────────────────────────────────
# codebuild_role = f"AmazonBedrockAgentCoreSDKCodeBuild-{region}-{agent_id.split('-')[-1]}"
# for policy in iam_client.list_attached_role_policies(RoleName=codebuild_role)["AttachedPolicies"]:
#     iam_client.detach_role_policy(RoleName=codebuild_role, PolicyArn=policy["PolicyArn"])
# iam_client.delete_role(RoleName=codebuild_role)
# print(f"✓ Deleted IAM role: {codebuild_role}")

print("To run cleanup: uncomment the relevant sections above and re-run this cell.")

## Summary


**Congratulations!** You've successfully deployed an enterprise-ready AI agent that combines Amazon Bedrock AgentCore with SAP GenAI Hub's governed AI consumption model.